# Ordinary Differential Equations — Interactive Notebook

Concept **29-ode** of the math-foundations learning map.

We will:
1. State the canonical IVP $y'=-y$, $y(0)=1$ and its exact solution.
2. Implement forward Euler from scratch (stdlib only).
3. Run Euler at several step sizes and tabulate the error.
4. Plot Euler trajectories against the exact $e^{-t}$ (matplotlib if available; ASCII fallback otherwise).
5. Verify the empirical $O(h)$ convergence order of Euler.
6. Connect to flow-matching samplers in modern ML.


## 1. The problem

$$y'(t) = -y(t), \qquad y(0) = 1, \qquad t \in [0, 5].$$

Exact solution: $y(t) = e^{-t}$. We treat this as ground truth.


In [ ]:
import math

f      = lambda t, y: -y            # vector field
exact  = lambda t   : math.exp(-t)  # closed-form solution

print('y(0) =', exact(0.0))
print('y(1) =', exact(1.0))
print('y(5) =', exact(5.0))


## 2. Forward Euler from scratch

$$y_{n+1} = y_n + h \cdot f(t_n, y_n), \qquad t_{n+1} = t_n + h.$$


In [ ]:
def euler(f, t0, y0, t_end, h):
    ts = [t0]
    ys = [y0]
    t, y = t0, y0
    n_steps = int(round((t_end - t0) / h))
    for _ in range(n_steps):
        y = y + h * f(t, y)
        t = t + h
        ts.append(t)
        ys.append(y)
    return ts, ys

ts, ys = euler(f, 0.0, 1.0, 5.0, 0.5)
for t, y in zip(ts, ys):
    print(f't={t:4.2f}  y_euler={y:8.5f}  y_exact={exact(t):8.5f}  err={abs(y-exact(t)):.3e}')


## 3. Error as a function of step size

Euler's global error is $O(h)$. Halving $h$ should roughly halve the max error.


In [ ]:
def max_abs_error(ts, ys, exact):
    return max(abs(y - exact(t)) for t, y in zip(ts, ys))

step_sizes = [0.5, 0.25, 0.1, 0.05, 0.01]
errors = []
print(f'{"h":>8} | {"steps":>6} | {"max |err|":>12}')
print('-' * 35)
for h in step_sizes:
    ts, ys = euler(f, 0.0, 1.0, 5.0, h)
    err = max_abs_error(ts, ys, exact)
    errors.append(err)
    print(f'{h:>8.3f} | {len(ts)-1:>6d} | {err:>12.6e}')


## 4. Trajectory plot (matplotlib if available, ASCII fallback)


In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np
    t_fine = np.linspace(0, 5, 400)
    plt.figure(figsize=(7, 4))
    plt.plot(t_fine, np.exp(-t_fine), 'k-', lw=2, label='exact e^{-t}')
    for h, marker in [(0.5, 'o'), (0.1, 's'), (0.01, '.')]:
        ts, ys = euler(f, 0.0, 1.0, 5.0, h)
        plt.plot(ts, ys, marker=marker, ms=4, lw=1, label=f'Euler h={h}')
    plt.xlabel('t'); plt.ylabel('y(t)')
    plt.title("Forward Euler vs exact for y' = -y")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.show()
except ImportError:
    # ASCII fallback (stdlib only)
    ts, ys = euler(f, 0.0, 1.0, 5.0, 0.25)
    width = 50
    for t, y in zip(ts, ys):
        bar = '#' * int(round(y * width))
        print(f't={t:4.2f} |{bar:<{width}}| y={y:.4f}')


## 5. Empirical order of convergence

If error $\sim C h^p$, then halving $h$ scales the error by $2^{-p}$. We expect $p \approx 1$ for Euler.


In [ ]:
for i in range(1, len(step_sizes)):
    h_prev, h_curr = step_sizes[i-1], step_sizes[i]
    e_prev, e_curr = errors[i-1], errors[i]
    ratio_h = h_prev / h_curr
    ratio_e = e_prev / e_curr
    order = math.log(ratio_e) / math.log(ratio_h)
    print(f'h: {h_prev:.3f} -> {h_curr:.3f}   error ratio = {ratio_e:7.3f}   order ~ {order:.3f}')


## 6. Connection to ML

In flow-matching / rectified-flow generative models (e.g. ELF), sampling integrates the learned vector field
$$\dot z_t = v_\theta(z_t, t)$$
from noise at $t=0$ to data at $t=1$. The default sampler is exactly the forward Euler step we wrote above, just in $\mathbb{R}^d$ with a neural network in place of $f$. The number of function evaluations (NFE) equals the number of Euler steps, and reducing $h$ trades compute for sample quality — exactly the trade-off we visualised here.
